## 사전 준비
이 노트북은 문서 임베딩에 로컬 Ollama 임베딩 모델 `qwen3-embedding:0.6b`를 사용합니다. 아래 명령어를 **터미널에서 미리 실행**해 모델을 받아두세요.

```
ollama pull qwen3-embedding:0.6b
```

# 로컬 LLM 기반의 RAG 챗봇 구현

## 실습 목표
---
로컬 임베딩 모델(Ollama)과 ChatGPT API를 결합한 RAG 챗봇을 구현합니다. 문서를 벡터화해 저장하고, 질문과 관련된 내용을 검색해 답변에 활용하는 전체 흐름을 익힙니다.

## 실습 목차
---

1. **OllamaEmbeddings를 활용한 문서 벡터화:** 로컬 임베딩 모델을 통해 문서를 Vector로 변환하기 위한 OllamaEmbeddings를 생성합니다. 생성한 임베딩을 통해 주어진 문서를 벡터화하여 Chroma DB에 저장합니다.

2. **Retriever Chain 구성:** 사용자의 입력과 가장 유사한 벡터화된 문서를 불러오는 Chain을 구성합니다.

3. **RAG Chain 구성:** RAG 기반 챗봇의 일부 기능을 구현한 미니 RAG Chain을 구성해봅시다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [1]:
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

/var/folders/px/v7_qzrl907d919ql0sn3f74r0000gn/T/ipykernel_20408/2030792407.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

- ChatGPT API(`.env`의 `MODEL_NAME`, 예: gpt-4o-mini)를 챗봇용으로, Ollama의 `qwen3-embedding:0.6b` 모델을 임베딩용으로 불러옵니다.
- 챗봇(생성)과 임베딩(문서 벡터화)은 서로 다른 역할이라, 하나의 모델로 통일할 필요 없이 각 역할에 맞는 모델을 따로 쓰는 것이 일반적입니다.

## 1. 시장조사 문서 벡터화
- RAG 챗봇에서 활용하기 위해 시장조사 파일을 읽어서 벡터화하는 과정을 실습합니다.

먼저, `.env`에 설정된 ChatGPT 모델을 사용하는 ChatOpenAI 객체와 `qwen3-embedding:0.6b` 모델을 사용하는 OllamaEmbeddings 객체를 생성합니다.

In [2]:
# llm: 사용자 질문에 답변을 생성하는 역할 (ChatGPT API)
# embeddings: 문서/질문을 벡터로 변환하는 역할 (로컬 Ollama 임베딩 모델)
llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")

`llm`과 `embeddings`가 각각 어떤 입력을 받아 어떤 출력을 내는지 간단히 확인해봅시다.

In [3]:
# ChatOpenAI: 메시지를 입력받아 답변 메시지를 생성합니다.
sample_response = llm.invoke("키오스크가 뭐야? 한 문장으로 답해줘.")
print(f"[llm 출력]\n{sample_response.content}")

# OllamaEmbeddings: 텍스트를 입력받아 숫자 벡터(리스트)로 변환합니다.
sample_vector = embeddings.embed_query("테스트 문장")
print(f"\n[embeddings 출력] 벡터 차원: {len(sample_vector)}, 앞 5개 값: {sample_vector[:5]}")

[llm 출력]
키오스크는 정보 제공이나 서비스 이용을 위해 사용자가 직접 조작할 수 있는 자동화된 기계입니다.

[embeddings 출력] 벡터 차원: 1024, 앞 5개 값: [0.011952761, -0.061316676, -0.011993341, -0.05455509, -0.0012392]


다음으로, 시장조사 PDF 문서를 불러와서 벡터화 해보겠습니다.
- 한국소비자원의 2022년 키오스크(무인정보단말기) 이용 실태조사 보고서를 활용했습니다
  - https://www.kca.go.kr/smartconsumer/sub.do?menukey=7301&mode=view&no=1003409523&page=2&cate=00000057
- 이 실태조사 보고서는 2022년 키오스크의 사용자 경험, 접근성, 후속 조치에 대해 논의하는 보고서입니다. 
- 이를 활용해서 키오스크를 어떻게 세일즈 할 수 있을지 아이디어를 제공하는 챗봇을 만들어야 하는 상황이라고 가정해 봅시다.

먼저, LangChain의 `PyPDFLoader`를 활용해서 시장조사 보고서의 텍스트를 추출하고, 페이지 별로 `Document`를 생성하여 저장합니다.

In [4]:
doc_path = "data/키오스크(무인정보단말기) 이용실태 조사.pdf"
loader = PyPDFLoader(doc_path)
docs = loader.load()

생성된 Document의 수와 각 Document 별 길이를 확인하는 함수를 정의하고, 불러온 보고서의 크기를 확인해 봅시다.

In [5]:
import statistics

def profile_docs(docs):
    doc_len = [len(doc.page_content) for doc in docs]
    print(f"Page 수: {len(docs)}")
    print(f"각 Document 별 길이: {doc_len}")
    print(f"평균 길이: {statistics.mean(doc_len):.2f}")
    print(f"중앙값: {statistics.median(doc_len)}")
    print(f"표준편차: {statistics.stdev(doc_len):.2f}")

In [6]:
profile_docs(docs)

Page 수: 59
각 Document 별 길이: [79, 6596, 5925, 5839, 1160, 861, 633, 1219, 787, 1073, 1381, 433, 978, 1128, 485, 1030, 918, 1034, 894, 1036, 935, 871, 1074, 815, 634, 1201, 1345, 1169, 657, 1060, 967, 522, 629, 855, 820, 1106, 465, 1159, 734, 594, 615, 722, 788, 375, 898, 811, 522, 936, 1145, 1081, 1172, 763, 507, 1208, 1175, 1631, 725, 751, 618]
평균 길이: 1144.81
중앙값: 898
표준편차: 1197.56


1천자 미만의 문서도 있지만, 6천자가 넘는 문서도 있는 것을 확인할 수 있습니다. 이대로 그냥 사용할 경우, Context가 너무 길어져 오히려 성능이 낮아질 수도 있습니다.

평균 길이(1145자)가 중앙값(899자)보다 큰 것을 보면, 몇 개의 유독 긴 문서가 평균을 끌어올렸다는 것을 알 수 있습니다. 이런 경우를 고려해서, 평균보다 조금 더 넉넉하게 1500자 단위로 나눠봅시다.

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
splited_docs = text_splitter.split_documents(docs)

In [8]:
profile_docs(splited_docs)

Page 수: 73
각 Document 별 길이: [79, 1500, 1498, 1499, 1498, 1395, 1499, 1498, 1499, 1499, 724, 1500, 1498, 1499, 1499, 639, 1160, 861, 633, 1219, 787, 1073, 1381, 433, 978, 1128, 485, 1030, 918, 1034, 894, 1036, 935, 871, 1074, 815, 634, 1201, 1345, 1169, 657, 1060, 967, 522, 629, 855, 820, 1106, 465, 1159, 734, 594, 615, 722, 788, 375, 898, 811, 522, 936, 1145, 1081, 1172, 763, 507, 1208, 1175, 35, 1397, 388, 725, 751, 618]
평균 길이: 960.51
중앙값: 936
표준편차: 367.63


Page 수가 많이 늘어난 대신, 1500자를 넘는 문서가 없는 것을 확인할 수 있습니다.

## 2. RAG 체인 구성
RAG 체인을 구성하기 위해 `Document`를 `OllamaEmbeddings`를 활용해 벡터로 변환하고, Chroma를 활용해 Vector DB로 변환하여 저장합니다.
- 변환 및 저장 과정은 약 2분 정도 소요됩니다.

In [9]:
%%time
vectorstore = Chroma.from_documents(documents=splited_docs, embedding=embeddings)

CPU times: user 112 ms, sys: 25.9 ms, total: 138 ms
Wall time: 25.8 s


In [10]:
# Chroma: 벡터와 원본 문서를 저장하고, 저장된 개수·내용을 조회할 수 있습니다.
print(f"저장된 벡터 수: {vectorstore._collection.count()}")

result = vectorstore._collection.get(include=["documents"])
print(f"첫 번째 문서 내용: \n{result['documents'][0][:100]}")

저장된 벡터 수: 73
첫 번째 문서 내용: 
키오스크(무인정보단말기) 이용 실태조사(담당자: 안세련 과장)
2022. 9.시 장 조 사 국시 장 감 시 팀
조사보회고서(이사회 요청 과제)


Vector DB를 `Retriever`로 변환합니다.
- `db_retriever`는 사용자의 질문(쿼리)을 임베딩 모델로 벡터화한 뒤, `vectorstore`에 저장된 문서 벡터들과 유사도를 계산하여 가장 유사한 문서들을 가져오는 역할을 합니다.

In [11]:
db_retriever = vectorstore.as_retriever()

`db_retriever`에 질문을 넣으면 어떤 문서가 검색되는지 확인해봅시다.

In [12]:
sample_docs = db_retriever.invoke("키오스크 이용률")
print(f"검색된 문서 수: {len(sample_docs)}")
print(f"첫 번째 문서 미리보기:\n{sample_docs[0].page_content[:150]}")

검색된 문서 수: 4
첫 번째 문서 미리보기:
키오스크(무인정보단말기)이용 실태조사
22 시장조사국 시장감시팀
Ⅴ소비자 설문조사▣ 설문대상: 키오스크 이용 경험이 있는 소비자 500명을 연령대별로 분류연령대20대(만 18세∼만 29세)30대(만 30세∼만 39세)40대(만 40세∼만 49세)50대(만 50세∼ 만 


Vector DB와 Retriever를 활용하는 RAG Chain을 구성합니다.
- `RunnablePassthrough()`는 Chain의 이전 구성 요소에서 전달된 값을 그대로 전달하는 역할을 수행합니다.

In [13]:
# ChatPromptTemplate: {context}, {question} 자리에 실제 값을 채워 최종 프롬프트(메시지 리스트)를 만듭니다.
# 이 llm은 "검색된 정보를 참고해 사용자 질문에 답하는" 역할을 맡습니다.
messages_with_contexts = [
    ("system", "당신은 마케터 지원 챗봇입니다. 주어진 정보를 참고해 질문에 답하세요."),
    ("human", "정보: {context}.\n{question}."),
]
prompt_with_context = ChatPromptTemplate.from_messages(messages_with_contexts)

sample_prompt = prompt_with_context.invoke({
    "context": "키오스크 이용률은 20대 78%, 60대 이상 36%입니다.",
    "question": "연령대별 이용률 알려줘",
})
print(sample_prompt.to_messages())

[SystemMessage(content='당신은 마케터 지원 챗봇입니다. 주어진 정보를 참고해 질문에 답하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content='정보: 키오스크 이용률은 20대 78%, 60대 이상 36%입니다..\n연령대별 이용률 알려줘.', additional_kwargs={}, response_metadata={})]


검색된 Document 리스트에서 텍스트만 뽑아내는 함수를 정의합니다. 어떤 노드가 실행됐는지, 몇 개 문서를 찾았는지 로그로 남깁니다.

In [14]:
import datetime

def get_retrieved_text(docs):
    print(f"[Retrieve] {len(docs)}개 문서 검색 완료")
    with open("retrieval_log.txt", "w", encoding="utf-8") as f:
        f.write(f"[{datetime.datetime.now()}] 검색된 문서 {len(docs)}개\n")
        for i, doc in enumerate(docs, 1):
            f.write(f"\n--- 문서 {i} ---\n{doc.page_content}\n")
    return "\n".join(doc.page_content for doc in docs)

sample_text = get_retrieved_text(sample_docs)
print(f"\n[get_retrieved_text 출력 미리보기]\n{sample_text[:150]}")

[Retrieve] 4개 문서 검색 완료

[get_retrieved_text 출력 미리보기]
키오스크(무인정보단말기)이용 실태조사
22 시장조사국 시장감시팀
Ⅴ소비자 설문조사▣ 설문대상: 키오스크 이용 경험이 있는 소비자 500명을 연령대별로 분류연령대20대(만 18세∼만 29세)30대(만 30세∼만 39세)40대(만 40세∼만 49세)50대(만 50세∼ 만 


StrOutputParser는 LLM이 반환하는 메시지 객체에서 텍스트(`content`)만 뽑아냅니다.

In [15]:
from langchain_core.messages import AIMessage

sample_output = StrOutputParser().invoke(AIMessage(content="이것은 예시 답변입니다."))
print(f"[StrOutputParser 출력] {sample_output}")

[StrOutputParser 출력] 이것은 예시 답변입니다.


RunnablePassthrough는 입력을 가공 없이 그대로 다음 단계로 전달합니다.

In [16]:
sample_passthrough = RunnablePassthrough().invoke("사용자 질문 그대로")
print(f"[RunnablePassthrough 출력] {sample_passthrough}")

[RunnablePassthrough 출력] 사용자 질문 그대로


지금까지 만든 조각들(prompt_with_context, get_retrieved_text, RunnablePassthrough, llm, StrOutputParser)을 이어붙여 RAG Chain을 완성합니다.

In [19]:
def init_chain():
    return (
        {"context": db_retriever | get_retrieved_text, "question": RunnablePassthrough()}
        | prompt_with_context
        | llm
        | StrOutputParser()
    )

qa_chain = init_chain()

## 3. 챗봇 구현 및 사용
- 구성한 RAG 체인을 활용해서 시장조사 문서 기반 챗봇을 구현하고 사용해봅니다.

방금 구현한 RAG Chain을 사용해서 시장조사 문서 기반 챗봇을 구현해볼 것입니다. 

그 전에, 별도로 RAG 기능을 추가하지 않은 LLM과 답변의 퀄리티를 비교해 봅시다.

In [17]:
# RAG를 사용하지 않은 기본 챗봇의 답변 (비교용)
messages_with_variables = [
    ("system", "당신은 마케터 지원 챗봇입니다."),
    ("human", "{question}."),
]
prompt = ChatPromptTemplate.from_messages(messages_with_variables)
chain = prompt | llm | StrOutputParser()

for chunk in chain.stream("키오스크 관련 설문조사 결과를 알려줘"):
    print(chunk, end="", flush=True)

키오스크 관련 설문조사 결과는 특정 조사에 따라 다를 수 있지만, 일반적으로 다음과 같은 주요 결과들이 나타납니다:

1. **사용 편의성**: 많은 사용자들이 키오스크의 사용 편의성을 높게 평가합니다. 특히, 직관적인 인터페이스와 빠른 결제 과정이 긍정적인 반응을 얻습니다.

2. **대기 시간 감소**: 키오스크를 이용함으로써 대기 시간이 줄어든다는 응답이 많습니다. 특히, 음식점이나 공항 등에서의 대기 시간을 단축시키는 데 효과적입니다.

3. **접근성**: 일부 사용자들은 키오스크의 접근성에 대해 우려를 표명하기도 합니다. 특히, 노인이나 기술에 익숙하지 않은 사용자들이 어려움을 겪는 경우가 있습니다.

4. **개인정보 보호**: 사용자들은 키오스크에서 개인정보 보호에 대한 우려를 가지고 있으며, 보안이 강화된 시스템을 선호합니다.

5. **결제 수단 다양성**: 다양한 결제 수단(신용카드, 모바일 결제 등)을 지원하는 키오스크에 대한 선호도가 높습니다.

6. **고객 서비스**: 키오스크 사용 후에도 고객 서비스와의 연결이 용이해야 한다는 의견이 많습니다. 문제가 발생했을 때 즉각적인 지원이 필요하다는 점이 강조됩니다.

이러한 결과들은 특정 산업이나 지역에 따라 다를 수 있으므로, 구체적인 설문조사 결과를 원하신다면 해당 조사 기관이나 연구 결과를 참고하는 것이 좋습니다.

별다른 출처를 추가하지 않은 챗봇은 알 수 없는 출처의 답변을 생성했습니다. 이제 RAG 챗봇에 동일한 질문을 입력해 봅시다.

In [20]:
# RAG를 사용한 챗봇의 답변
for chunk in qa_chain.stream("키오스크 관련 설문조사 결과를 알려줘"):
    print(chunk, end="", flush=True)

[Retrieve] 4개 문서 검색 완료
키오스크(무인정보단말기) 이용 실태조사 결과는 다음과 같습니다:

### 1. 이용 빈도
- 연령대가 높아질수록 키오스크 이용 빈도가 낮아짐.
- 20대(78.0%)와 30대(76.0%)는 1주일에 1회 이상 이용한다고 응답한 비중이 높았으나, 60대 이상은 36.0%에 불과함.

### 2. 이용 이유 및 업종
- 키오스크를 이용하는 주요 이유는 '키오스크로만 주문이 가능'함.
- 가장 많이 이용한 업종은 '외식업', '유통점포', '주차장'이며, 60대 이상은 '병원' 이용 비중이 높음.

### 3. 이용 수월성
- 업종별 이용 수월성은 '무인 점포'가 가장 높고, '관공서'가 가장 낮음.

### 4. 거래 방식 선호도
- 20대~50대는 비대면 거래를 선호하고, 60대 이상은 대면 거래를 선호함.

### 5. 키오스크 개선 필요성
- **기기 자체 개선**: 모든 연령대가 필요성을 느끼며, 특히 60대 이상이 높게 평가함.
- **기능 표준화**: 모든 연령대가 선호하며, 60대 이상의 선호도가 특히 높음.
- **교육 확대**: 60대 이상이 가장 높은 선호도를 보이며, 81%가 필요하다고 응답함.
- **배려 존 마련**: 모든 연령대가 긍정적으로 평가함.

### 6. 편의성 및 접근성 개선
- 모든 연령대가 화면의 특정 부분을 터치하도록 유도하는 초점 표시를 선호함.
- 추천 상품 알려주기 기능은 20대~40대는 불편하다고 느끼고, 50대~60대 이상은 편리하다고 느낌.
- 직원 배치 또는 호출벨 설치는 모든 연령대가 선호함.

### 7. 교육 참여 의사
- 20대~50대는 '현재도 충분히 잘 이용하고 있어서'라는 이유로 교육 참여 의사가 낮고, 60대 이상은 '흥미가 없어서', '효과가 없을 것 같아서' 등의 이유로 다양하게 나타남.

이 조사 결과는 키오스크의 이용 실태와 소비자들의 요구를 파악하는 데 중요한 정보를 제공합니다.

일반 체인은 아무런 출처가 없는 답변을 생성한 반면, RAG 기능을 추가한 챗봇은 데이터를 기반으로 상대적으로 정확한 답변을 하는 것을 확인할 수 있습니다.

이제 챗봇을 한번 사용해 봅시다.

In [25]:
qa_chain = init_chain()
while True:
    question = input("질문을 입력해주세요 (종료를 원하시면 '종료'를 입력해주세요.): ")
    if question == "종료":
        break
    else:
        # invoke() 대신 stream()을 사용하면 토큰이 생성되는 대로 바로 출력할 수 있습니다.
        for chunk in qa_chain.stream(question):
            print(chunk, end="", flush=True)
        print()

[Retrieve] 4개 문서 검색 완료
[Generate] 답변 생성 중... (retrieval_log.txt에서 검색된 문서 전체를 확인할 수 있습니다)
20대는 78%, 30대는 76%, 40대는 62%, 50대는 45%, 60대 이상은 36%로, 연령이 높아질수록 키오스크의 이용률이 낮은 것으로 나타났습니다.
